In [2]:
""" Importing necessary modules """

from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

In [3]:
# Use a pipeline as a high-level helper

pipe = pipeline("text-generation", model="LiquidAI/LFM2-350M-Extract")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

Device set to use cpu


[{'generated_text': [{'role': 'user', 'content': 'Who are you?'},
   {'role': 'assistant',
    'content': '{\n  "name": "Giovanni",\n  "birth_departure": "Florence",\n  "parents": "Marco",\n  "education": "Bachelor",\n  "passport_number": "3",\n  "passport_type": "A",\n  "passport_issue_date": "1965-01-01",\n  "passport_expiry_date": "1966-01-01",\n  "nationality": "Italy",\n  "nationality_affiliation": "Italy",\n  "biometric_data": {\n    "fingerprint": "N/A",\n    "iris_scan": "N/A",\n    "facial_recognition": "N/A"\n  },\n  "personal_information": {\n    "date_of_birth": "1945-03-15",\n    "gender": "Female",\n    "marital_status": " remarried",\n    "nationality_affiliation": "N/A"\n  },\n  "occupation": "Artisan",\n  "occupation_affiliation": "N/A",\n  "education_degree": "MBA",\n  "education_degree": "MBA",\n  "passport_number_2": "4",\n '}]}]

In [9]:
# Load model directly

tokenizer = AutoTokenizer.from_pretrained("LiquidAI/LFM2-350M-Extract")
model = AutoModelForCausalLM.from_pretrained("LiquidAI/LFM2-350M-Extract")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)


In [1]:
from Modules.Models.liquid import liquid_desc_to_json

print(liquid_desc_to_json('Teste'))

{
  "teste": "Teste",
  "data": "data",
  "version": "1.2.3",
  "release_details": {
    "release_date": "2024-01-26",
    "release_description": "New features and improvements",
    "new_features": [
      {
        "feature_name": "New Speed",
        "description": "New Speed",
        "priority": "High"
      },
      {
        "feature_name": "Improved Responsiveness",
        "description": "Improved Responsiveness",
        "priority": "Medium"
      },
      {
        "feature_name": "Faster Load Time",
        "description": "Faster Load Time",
        "priority


---

In [5]:
""" Decomposing gradually """
inputs_ids = inputs["input_ids"]
inputs_ids_shapes = inputs_ids.shape
last_shape = inputs_ids_shapes[-1]
first_output = outputs[0]

to_decode = first_output[last_shape:]
decoded = tokenizer.decode(to_decode)

print('inputs: ', inputs)
print('inputs_ids: ', inputs_ids)
print('inputs_ids_shapes: ', inputs_ids_shapes)
print('last_shape: ', last_shape)
print('outputs: ', outputs)
print('first_output: ', first_output)
print('to_decode: ', to_decode)
print('decoded: ', decoded)

# print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))


inputs:  {'input_ids': tensor([[    1,     6,  6423,   708, 17408,   938,  1010,   540,     7,   708,
             6, 64015,   708]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
inputs_ids:  tensor([[    1,     6,  6423,   708, 17408,   938,  1010,   540,     7,   708,
             6, 64015,   708]])
inputs_ids_shapes:  torch.Size([1, 13])
last_shape:  13
outputs:  tensor([[    1,     6,  6423,   708, 17408,   938,  1010,   540,     7,   708,
             6, 64015,   708,  3219,   730,   997,  3055,  6881,   997,   550,
          1283,   768,  5354,  6608,   730,   997,  1047,  6881,   730,  1374,
          1421,   730,   997, 35014,  6881,   997, 43706,  6608,   730,   997,
         12451,  6881,   730,   526,   523,   530,  1421,   730,   997,  9497,
          6881,   730,   526]])
first_output:  tensor([    1,     6,  6423,   708, 17408,   938,  1010,   540,     7,   708,
            6, 64015,   708,  3219,   730,   997,  3055,  6881,   997,   550,
        

In [6]:
%pip install -q hjson json5 demjson3 dirtyjson pyyaml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
""" Clean broken dictionaries and convert to valid JSON """

broken_jsons = [
    '{"a": 1', # missing closing brace
    '{"a": {"b": 1', # missing closing braces
    "{'a': {'b': 1", # missing closing braces
    '{"a": [1', # Missing closing brackets
    '{"a": [1,', # Missing closing brackets
]


def clean_broken_json(s: str) -> dict|None:
    """
    Parse a broken JSON-like string trying parsers in the order given by
    parser_sequence. parser_sequence may be:
      - None (use default order)
      - list of parser names (strings)
      - list of callables (callable(s) taking the string and returning a dict)
    Example:
      clean_broken_json(s, parser_sequence=['my_custom', 'json', 'ast'])
      clean_broken_json(s, parser_sequence=[my_custom_parser, json.loads])
    """
    import json
    import ast
    import hjson
    import json5
    import demjson3
    import dirtyjson
    import yaml

    def check_none(value):
        if value is None:
            raise ValueError('None value')

    
    # tentativa 1: json padrão
    cleaned = None
    
    try:
        cleaned = json.loads(s)
        print('JSON: ', cleaned)
        check_none(cleaned)
        return cleaned
    except Exception:
        pass
    # tentativa 2: literal Python
    try:
        cleaned = ast.literal_eval(s)
        print('ast: ', cleaned)
        check_none(cleaned)
        return cleaned
    except Exception:
        pass
    # tentativa 3+: bibliotecas relaxadas (instalar: hjson, json5, demjson3, dirtyjson)
    try:
        cleaned = hjson.loads(s)
        print('hjson: ', cleaned)
        check_none(cleaned)
        return cleaned
    except Exception:
        pass
    try:
        cleaned = json5.loads(s)
        print('json5: ', cleaned)
        check_none(cleaned)
        return cleaned
    except Exception:
        pass
    try:
        cleaned = demjson3.decode(s)
        print('demjson3: ', cleaned)
        check_none(cleaned)
        return cleaned
    except Exception:
        pass
    try:
        cleaned = dirtyjson.loads(s)
        print('dirtyjson: ', cleaned)
        check_none(cleaned)
        return cleaned
    except Exception:
        pass
    # fallback simples: trocar aspas simples por duplas e fechar chaves (pode alterar significado)
    t = s.replace("'", '"')
    # balancear parênteses/brackets simples (exemplo mínimo)
    opens = t.count("{") - t.count("}")
    t = t + ("}" * opens) if opens > 0 else t
    try:
        return json.loads(t)
    except Exception:
        return None

for broken_dict in broken_jsons:
    cleaned_dict = clean_broken_json(broken_dict)
    print('Broken:', broken_dict)
    print('Cleaned:', cleaned_dict)
    print('---')

Broken: {"a": 1
Cleaned: {'a': 1}
---
Broken: {"a": {"b": 1
Cleaned: {'a': {'b': 1}}
---
Broken: {'a': {'b': 1
Cleaned: {'a': {'b': 1}}
---
Broken: {"a": [1
Cleaned: None
---
Broken: {"a": [1,
Cleaned: None
---


In [8]:
""" Clean broken dictionaries and convert to valid JSON """

import json
import ast
import hjson
import json5
import demjson3
import dirtyjson
import yaml
import re

broken_jsons = [
    '{"a": 1',  # missing closing brace
    '{"a": {"b": 1',  # missing closing braces
    "{'a': {'b': 1",  # missing closing braces (single quotes)
    '{"a": [1',  # Missing closing brackets
    '{"a": [1,',  # Missing closing brackets + trailing comma
    '{"a": [{"b": 1, {"c":2}]',  # more complex broken (for teste)
]


def clean_broken_json(s: str) -> dict|None:
    """
    Parse a broken JSON-like string trying parsers in the order given by
    parser_sequence. parser_sequence may be:
      - None (use default order)
      - list of parser names (strings)
      - list of callables (callable(s) taking the string and returning a dict)
    Example:
      clean_broken_json(s, parser_sequence=['my_custom', 'json', 'ast'])
      clean_broken_json(s, parser_sequence=[my_custom_parser, json.loads])
    """

    parsers = [
        json.loads,
        ast.literal_eval,
        hjson.loads,
        json5.loads,
        demjson3.decode,
        dirtyjson.loads,
        yaml.safe_load
    ]
    
    for parser in parsers:
        try:
            cleaned = parser(s)
            return cleaned
        except Exception:
            pass

    t = s.replace("'", '"')  # troca aspas simples
    # remove vírgulas antes de chaves/colchetes de fechamento
    t = re.sub(r",(\s*[}\]])", r"\1", t)

    # Balancear chaves e colchetes
    braces_diff = t.count("{") - t.count("}")
    brackets_diff = t.count("[") - t.count("]")

    if braces_diff > 0:
        t += "}" * braces_diff
    if brackets_diff > 0:
        t += "]" * brackets_diff

    try:
        return json.loads(t)
    except Exception:
        return json.loads({})



for broken_dict in broken_jsons:
    cleaned_dict = clean_broken_json(broken_dict)
    print('Broken:', broken_dict)
    print('Cleaned:', cleaned_dict)
    print('---')

Broken: {"a": 1
Cleaned: {'a': 1}
---
Broken: {"a": {"b": 1
Cleaned: {'a': {'b': 1}}
---
Broken: {'a': {'b': 1
Cleaned: {'a': {'b': 1}}
---


TypeError: the JSON object must be str, bytes or bytearray, not dict